In [18]:
import numpy as np
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DATA KOMPONEN MURNI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

pure_components = {
    #   nama                  κAB              εAB/K
    "glycine"      : (0.040214817, 2552.247118),
    "alanine"      : (0.050694465, 3345.982941),
    "valine"       : (0.035670721, 3469.042435),
    "leucine"      : (0.035713577, 3667.379487),
    "serine"       : (0.032905355, 2086.932849),
    
}

molecules = ["glycine", "alanine", "valine", "leucine", "serine"]


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — PARAMETER REFERENSI GRUP (INPUT — SUDAH DIKETAHUI)
# ══════════════════════════════════════════════════════════════════════════════

GC_ref = {
    #  grup     κAB_ref      εAB_ref/K
    "CH3"  : (0.0,          0.0      ),   
    "CH2"  : (0.0,          0.0      ),  
    "CH"   : (0.0,          0.0      ),  
    "NH2"  : (0.149350,     504.320  ),   
    "OH"   : (0.001341,     2217.87  ),   
    "COOH" : (-0.012292,  2676.778915),  
   
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — PARAMETER PERTURBATIF μ YANG SUDAH DIKETAHUI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

mu_known = {

    "CH3-CH3"  : {"kap":  0.0,   "eps":  0.0  },  
    "CH3-CH"   : {"kap":  0.0,   "eps":  0.0  },   
    "CH2-CH3"  : {"kap":  0.0,   "eps":  0.0  },   
    "CH2-CH2"  : {"kap":  0.0,   "eps":  0.0  },
    "CH2-CH"   : {"kap":  0.0,   "eps":  0.0  },   
    "CH-CH"   : {"kap":  0.0,   "eps":  0.0  }, 
    
    "CH3-NH2"  : {"kap":  -0.0311,   "eps":  111.93  },   
    "CH2-NH2"  : {"kap":  0.0,   "eps":  0.0  },   
    "CH-NH2"   : {"kap":  -0.098681,   "eps":  120.102211},
    
    "CH3-OH"   : {"kap":  0.0484919,   "eps":  619.549},
    "CH2-OH"   : {"kap":  0.0,   "eps":  0.0  },
    "CH-OH"    : {"kap":  -0.029826,   "eps": -422.509509},
    
    "CH3-COOH" : {"kap": 0.129695, "eps":-239.617839},
    "CH2-COOH" : {"kap":  0.0,   "eps":  0.0  },
    "CH-COOH"  : {"kap":  -0.115487,   "eps": 82.499712},

    "NH2-OH"   : {"kap": -0.084806,   "eps":  -681.28663},
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — S-TERM (INPUT MANUAL)
# ══════════════════════════════════════════════════════════════════════════════

S_terms = {
    #             ["glycine", "alanine", "valine", "leucine", "serine"]
    "CH3-CH3" : [0, 0, 0.5, 0.5, 0],
    "CH3-CH" : [0, 1, 3, 2.66666666666667, 0],
    "CH2-CH3" : [0, 0, 0, 1, 0],
    
    "CH2-CH2" : [0, 0, 0, 0, 0],
    "CH2-CH" : [0, 0, 0, 2, 1],

    "CH-CH" : [0, 0, 1, 0.5, 0],

    "CH3-NH2" : [0, 0.5, 0.666666666666667, 0.5, 0],
    "CH2-NH2" : [1, 0, 0, 0.5, 0.5],
    "CH-NH2" : [0, 1, 1.5, 1.33333333333333, 1],
 
    "CH3-OH" : [0, 0, 0, 0, 0],
    "CH2-OH" : [0, 0, 0, 0, 1],
    "CH-OH" : [0, 0, 0, 0, 0.5],
 
    "CH3-COOH" : [0, 0.5, 0.666666666666667, 0.5, 0],
    "CH2-COOH" : [1, 0, 0, 0.5, 0.5],
    "CH-COOH" : [0, 1, 1.5, 1.33333333333333, 1],
 
    "NH2-OH" : [0, 0, 0, 0, 0.333333333333333],
    "NH2-COOH" : [0.5, 0.5, 0.5, 0.5, 0.5],
    "COOH-OH" : [0, 0, 0, 0, 0.333333333333333],
    
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — VARIABEL YANG DI-FITTING (UNKNOWN)
# ══════════════════════════════════════════════════════════════════════════════

unknown_pairs = [
    "NH2-COOH",
    "COOH-OH"  
    
]

# Jumlah unknown per properti (dihitung otomatis)
N_unk = len(unknown_pairs)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PERSAMAAN MODEL PER MOLEKUL
# ══════════════════════════════════════════════════════════════════════════════


def model(mol_idx, prop, unk):
    mu_NH2_COOH = unk[0]    
    mu_COOH_OH = unk[1]
    
    if mol_idx == 0:
        #glycine
        return (
            0 * gc("CH3", prop)
            + 1 * gc("CH2", prop)
            + 0 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 1 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 1 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 1 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 1 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 1 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )
        
        
    elif mol_idx == 1:
        # alanine
         return (
            1 * gc("CH3", prop)
            + 0 * gc("CH2", prop)
            + 1 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 1 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 1 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 1 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 1 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 1 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    elif mol_idx == 2:
        # valine
         return (
            2 * gc("CH3", prop)
            + 0 * gc("CH2", prop)
            + 2 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 1 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 1 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 1 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 1 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 1 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    elif mol_idx == 3:
        # leucine
         return (
            2 * gc("CH3", prop)
            + 1 * gc("CH2", prop)
            + 2 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 0 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 1 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 1 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 1 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 1 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 1 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    elif mol_idx == 4:
        # serine
         return (
            0 * gc("CH3", prop)
            + 1 * gc("CH2", prop)
            + 2 * gc("CH", prop)
            + 1 * gc("NH2", prop)
            + 1 * gc("COOH", prop)
            + 1 * gc("OH", prop)

            + 1 * mk("CH3-CH3", prop) * S("CH3-CH3", mol_idx)
            + 1 * mk("CH3-CH", prop) * S("CH3-CH", mol_idx)
            
            + 1 * mk("CH2-CH3", prop) * S("CH2-CH3", mol_idx)
            + 1 * mk("CH2-CH2", prop) * S("CH2-CH2", mol_idx)
            + 1 * mk("CH2-CH", prop) * S("CH2-CH", mol_idx)
            
            + 1 * mk("CH3-NH2", prop) * S("CH3-NH2", mol_idx)
            + 1 * mk("CH2-NH2", prop) * S("CH2-NH2", mol_idx)
            + 1 * mk("CH-NH2", prop) * S("CH-NH2", mol_idx)

            + 1 * mk("CH3-OH", prop) * S("CH3-OH", mol_idx)
            + 1 * mk("CH2-OH", prop) * S("CH2-OH", mol_idx)
            + 1 * mk("CH-OH", prop) * S("CH-OH", mol_idx)

            + 1 * mk("CH3-COOH", prop) * S("CH3-COOH", mol_idx)
            + 1 * mk("CH2-COOH", prop) * S("CH2-COOH", mol_idx)
            + 1 * mk("CH-COOH", prop) * S("CH-COOH", mol_idx)

            + 2 * mk("NH2-OH", prop) * S("NH2-OH", mol_idx)
            
            + 2 * mu_NH2_COOH * S("NH2-COOH", mol_idx)
            + 2 * mu_COOH_OH * S("COOH-OH", mol_idx)
        )

    else:
        raise ValueError(f"mol_idx={mol_idx} tidak ada dalam daftar molekul.")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — BATAS (BOUNDS) UNTUK UNKNOWN
# ══════════════════════════════════════════════════════════════════════════════

bounds = [
   #kappa
    (0.13,  0.15), (-1., 1.0),
    #epsilon
    (-6000.0,  6000.0), (-6000.0, 6000.0),
]

assert len(bounds) == N_unk * 2, (
    f"Jumlah bounds ({len(bounds)}) harus = N_unk×2 = {N_unk*2}. "
    f"Sesuaikan SECTION 7: {N_unk} baris untuk κAB, lalu {N_unk} baris untuk εAB."
)


# ══════════════════════════════════════════════════════════════════════════════
# Helper functions, objective function, optimizer, output
# ══════════════════════════════════════════════════════════════════════════════

# ── Helper: ambil nilai referensi grup ──────────────────────────────────────
def gc(group, prop):
    """
    Kembalikan nilai referensi κAB atau εAB untuk grup.
    prop = "kap" → κAB_grup
    prop = "eps" → εAB_grup
    """
    kap_g, eps_g = GC_ref[group]
    return {"kap": kap_g, "eps": eps_g}[prop]

# ── Helper: ambil μ known ────────────────────────────────────────────────────
def mk(pair, prop):
    """Kembalikan nilai μ yang sudah diketahui untuk pasangan dan properti."""
    if pair not in mu_known:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di mu_known (SECTION 3). "
            f"Tambahkan jika diketahui, atau masukkan ke unknown_pairs (SECTION 5)."
        )
    return mu_known[pair][prop]

# ── Helper: ambil S-term ─────────────────────────────────────────────────────
def S(pair, mol_idx):
    """Kembalikan nilai S-term untuk pasangan dan indeks molekul."""
    if pair not in S_terms:
        raise KeyError(
            f"Pasangan '{pair}' tidak ada di S_terms (SECTION 4). "
            f"Tambahkan nilainya."
        )
    return S_terms[pair][mol_idx]

# ── Hitung target dari data komponen murni ───────────────────────────────────
# targets["kap"] = array κAB tiap molekul
# targets["eps"] = array εAB tiap molekul
targets = {"kap": [], "eps": []}
for mol in molecules:
    kap, eps = pure_components[mol]
    targets["kap"].append(kap)
    targets["eps"].append(eps)
targets = {prop: np.array(v) for prop, v in targets.items()}

# ── Unpack unknown dari vektor x ─────────────────────────────────────────────
def unpack(x, prop):
    """
    Ambil slice unknown untuk properti tertentu dari vektor x optimizer.
    Urutan dalam x: [kap_unk0, kap_unk1, ..., eps_unk0, eps_unk1, ...]
    """
    i = {"kap": 0, "eps": 1}[prop]
    return x[i * N_unk : (i + 1) * N_unk]

# ── Objective function ────────────────────────────────────────────────────────

def objective(x):
    
    F = 0.0
    for prop in ("kap", "eps"):
        unk = unpack(x, prop)
        for mol_idx in range(len(molecules)):
            pi_GC  = model(mol_idx, prop, unk)
            pi_tgt = targets[prop][mol_idx]
            # Hindari pembagian dengan nol jika target = 0
            if abs(pi_tgt) > 1e-15:
                F += ((pi_tgt - pi_GC) / pi_tgt) ** 2
    return F


# ── Optimasi dua tahap: global → lokal ───────────────────────────────────────
print("=" * 62)
print("  PC-SAFT PertGC — Association Parameter Fitting Template")
print("=" * 62)
print(f"\n  Molekul  : {', '.join(molecules)}")
print(f"  Unknown  : {', '.join(unknown_pairs)}")
print(f"  Total    : {N_unk} unknown × 2 properti = {N_unk*2} parameter\n")

print("  Stage 1: Global search (differential_evolution)...")

res_g = differential_evolution(
    objective, bounds,
    seed=42, maxiter=8000, tol=1e-14,
    popsize=25, mutation=(0.5, 1.5), recombination=0.9,
    polish=True, disp=False,
)


print("  Stage 2: Local refinement (L-BFGS-B)...")
res_l = minimize(
    objective, res_g.x,
    method="L-BFGS-B", bounds=bounds,
    options={"maxiter": 100000, "ftol": 1e-16, "gtol": 1e-13},
)

x_opt = res_l.x if res_l.fun < res_g.fun else res_g.x
F_opt = min(res_l.fun, res_g.fun)

# ── Cetak parameter hasil fitting ────────────────────────────────────────────
prop_label = {"kap": "κAB", "eps": "εAB (K)"}

print("\n" + "=" * 62)
print("  FITTED PARAMETERS")
print("=" * 62)
for prop in ("kap", "eps"):
    unk = unpack(x_opt, prop)
    print(f"\n  Properti: {prop_label[prop]}")
    for j, pair in enumerate(unknown_pairs):
        print(f"    μ_{prop_label[prop].split()[0]}({pair:<18}) = {unk[j]:>16.6f}")

print(f"\n  F_OBJ (final) = {F_opt:.6e}")

# ── Validasi: target vs. prediksi GC ─────────────────────────────────────────
print("\n" + "=" * 62)
print("  VALIDATION — Target vs. GC-Predicted (APD%)")
print("=" * 62)

for prop in ("kap", "eps"):
    unk = unpack(x_opt, prop)
    print(f"\n  {prop_label[prop]}")
    print(f"  {'Molekul':<20} {'Target':>14} {'GC-Pred':>14} {'APD%':>9}")
    print(f"  {'-'*60}")
    for mol_idx, mol in enumerate(molecules):
        pi_tgt = targets[prop][mol_idx]
        pi_gc  = model(mol_idx, prop, unk)
        apd    = 100.0 * abs(pi_tgt - pi_gc) / abs(pi_tgt) if abs(pi_tgt) > 1e-15 else 0.0
        print(f"  {mol:<20} {pi_tgt:>14.6f} {pi_gc:>14.6f} {apd:>8.4f}%")

# ── Tabel ringkasan ───────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  SUMMARY TABLE (copy-paste ready)")
print("=" * 62)
print(f"  {'Parameter':<28} {'κAB':>14} {'εAB (K)':>14}")
print("  " + "-" * 58)
for j, pair in enumerate(unknown_pairs):
    val_kap = unpack(x_opt, "kap")[j]
    val_eps = unpack(x_opt, "eps")[j]
    print(f"  μ({pair:<24}) {val_kap:>14.6f} {val_eps:>14.4f}")

# ── Diagnostik batas ──────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  DIAGNOSTIC — Bound Check")
print("=" * 62)

all_labels = (
    [f"μ_kap({p})" for p in unknown_pairs] +
    [f"μ_eps({p})" for p in unknown_pairs]
)

bound_hit = False
for i, (val, (lo, hi)) in enumerate(zip(x_opt, bounds)):
    tol = max(1e-4, 1e-4 * abs(hi - lo))
    if abs(val - lo) < tol or abs(val - hi) < tol:
        print(f"  ⚠  {all_labels[i]} = {val:.6f} menempel di batas [{lo}, {hi}].")
        print(f"      → Perlebar batas ini di SECTION 7 dan jalankan ulang.")
        bound_hit = True
if not bound_hit:
    print("  ✓  Tidak ada parameter yang menempel di batas. Solusi interior.")

  PC-SAFT PertGC — Association Parameter Fitting Template

  Molekul  : glycine, alanine, valine, leucine, serine
  Unknown  : NH2-COOH, COOH-OH
  Total    : 2 unknown × 2 properti = 4 parameter

  Stage 1: Global search (differential_evolution)...
  Stage 2: Local refinement (L-BFGS-B)...

  FITTED PARAMETERS

  Properti: κAB
    μ_κAB(NH2-COOH          ) =         0.130000
    μ_κAB(COOH-OH           ) =         0.075187

  Properti: εAB (K)
    μ_εAB(NH2-COOH          ) =      -168.981201
    μ_εAB(COOH-OH           ) =     -4020.316018

  F_OBJ (final) = 3.338252e+01

  VALIDATION — Target vs. GC-Predicted (APD%)

  κAB
  Molekul                      Target        GC-Pred      APD%
  ------------------------------------------------------------
  glycine                    0.040215       0.267058 564.0786%
  alanine                    0.050694       0.102188 101.5753%
  valine                     0.035671       0.011536  67.6598%
  leucine                    0.035714       0.030798 